# Phase 4 — Cross-Jurisdiction Equivalency Graph

Interactive walkthrough of the cross-jurisdiction equivalency graph.
Per the multi-stage plan (see AGENTS.md).

Reads every topic from the Phase 3 ``extracted_syllabi`` SQLite table,
calls BAML ``ExtractEquivalencies`` for each topic, and builds a
topic-node + equivalency-edge graph in dev SQLite.

In production (FalkorDB), the same tables are mirrored to a
``:TopicNode`` + ``:TopicEquivalentEdge`` graph via the canonical
CocoIndex FalkorDB connector.

In [ ]:
# 1. Check the Phase 3 SQLite state.
import pathlib
import sqlite3

SQLITE = pathlib.Path("data/bi_ep/extracted_syllabi.sqlite")
print(f"DB exists: {SQLITE.exists()}")
if SQLITE.exists():
    with sqlite3.connect(str(SQLITE)) as conn:
        rows = conn.execute(
            "SELECT subnation, subject_slug, language, "
            "substr(syllabus_json, 1, 80) "
            "FROM extracted_syllabi"
        ).fetchall()
    print(f"Source rows: {len(rows)}")
    for r in rows[:5]:
        print(f"  {r[0]:20s} {r[1]:20s} {r[2]:3s}  {r[3]}...")
else:
    print("No Phase 3 DB yet. Run Phase 3 first:")
    print(
        "  python -m cocoindex_flows.education.lc6_extraction_app --subject mathematics --language en"
    )

In [ ]:
# 2. Build the equivalency graph.
from cocoindex_flows.equivalency.equivalency_graph_app import build_equivalency_graph

stats = build_equivalency_graph()
print(stats)

In [ ]:
# 3. Inspect the resulting nodes + edges.
if SQLITE.exists():
    with sqlite3.connect(str(SQLITE)) as conn:
        nodes = conn.execute(
            "SELECT subnation, topic_name, topic_key "
            "FROM topic_nodes ORDER BY subnation, topic_name"
        ).fetchall()
        edges = conn.execute(
            "SELECT source_subnation, source_topic_name, "
            "target_subnation, target_topic_name, confidence "
            "FROM topic_equivalent_edges "
            "ORDER BY source_topic_name, target_subnation"
        ).fetchall()
    print(f"=== {len(nodes)} topic nodes ===")
    for n in nodes[:10]:
        print(f"  {n[0]:18s}  {n[1]:40s}  key:{n[2]}")
    print(f"\n=== {len(edges)} equivalency edges ===")
    for e in edges[:15]:
        print(f"  {e[0]:18s}  '{e[1]}' -> {e[2]:18s}  '{e[3]}'  ({e[4]:.2f})")

In [ ]:
# 4. Visualize the equivalency graph (text-mode for the demo).
from collections import defaultdict

if SQLITE.exists():
    with sqlite3.connect(str(SQLITE)) as conn:
        rows = conn.execute(
            "SELECT source_topic_name, target_subnation, target_topic_name, confidence "
            "FROM topic_equivalent_edges WHERE confidence >= 0.7"
        ).fetchall()

    by_source = defaultdict(list)
    for src, tgt_j, tgt_n, conf in rows:
        by_source[src].append((tgt_j, tgt_n, conf))

    for src in sorted(by_source.keys()):
        print(f"\n  {src}")
        for tgt_j, tgt_n, conf in sorted(by_source[src]):
            print(f"    --{conf:.2f}-->  {tgt_j:18s}  '{tgt_n}'")

## Summary

- The equivalency graph is built by reading topics from Phase 3 + calling BAML ``ExtractEquivalencies``.
- When ``baml_client`` is importable, BAML produces real cross-jurisdiction mappings. When missing, a stub returns identity equivalence (with ``stub: True``).
- Storage: 2 SQLite tables (``topic_nodes`` + ``topic_equivalent_edges``) — composite primary keys, upsert semantics.
- Production path (Phase 8): mirror to FalkorDB via the CocoIndex FalkorDB connector. Same schema; different storage.
- Ready for Phase 5 (model comparison harness) — the equivalency graph is one of the comparison surfaces, ( cost / time / accuracy across 5 models).